In [0]:
dados = [("Databricks Serveless", 1), ("Engenharia de Dados", 2)]

In [0]:
df = spark.createDataFrame(dados, ["Ambiente", "ID"])    

In [0]:
display(df)

Ambiente,ID
Databricks Serveless,1
Engenharia de Dados,2


In [0]:
url_dados_abertos = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/flights.csv"

In [0]:
df_bronze = spark.read.option("header", "true").option("inferSchema", "true").csv((url_dados_abertos))

In [0]:
nome_tabela_bronze = "bronze_transporte"

In [0]:
df_bronze.write.format("delta").mode("overwrite").saveAsTable(nome_tabela_bronze)

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-7227616352490455>, line 1
----> 1 df_bronze.write.format("delta").mode("overwrite").saveAsTable(nome_tabela_bronze)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1556, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1554     req.user_context.user_id = self._user_id
   1555 self._set_command_in_plan(req.pla

In [0]:
import pandas as pd

# 1. Baixando os dados brutos da web via Pandas (simulando a chegada do dado bruto da API)
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/flights.csv"
pdf = pd.read_csv(url)

# 2. Convertendo para DataFrame do PySpark
df_bronze = spark.createDataFrame(pdf)

# 3. Gravando na Camada Bronze como tabela Delta Lake
nome_tabela_bronze = "bronze_transporte"
df_bronze.write.format("delta").mode("overwrite").saveAsTable(nome_tabela_bronze)

# 4. Verificando o resultado no Lakehouse
print("Camada Bronze criada com sucesso no Delta Lake!")
display(spark.sql(f"SELECT * FROM {nome_tabela_bronze} LIMIT 5"))

Camada Bronze criada com sucesso no Delta Lake!


year,month,passengers
1949,January,112
1949,February,118
1949,March,132
1949,April,129
1949,May,121


In [0]:
from pyspark.sql.functions import col, when, current_timestamp

df_raw = spark.read.table("bronze_transporte")

In [0]:
df_silver = (
    df_raw
    .withColumnRenamed("year", "ano")
    .withColumnRenamed('month', "mes")
    .withColumnRenamed("passagers", "qtd_passageiros")
    .dropna(subset=["ano", "mes", "qtd_passageiros"])
    .dropDuplicates()
    .withColumn(
        "categoria_demanda",
        when(col("qtd_passageiros") >= 300, "Alta")
        .when(col("qtd_passageiros") >= 150, "Média")
        .otherwise("Baixa")
    )
    .withColumn("data_processamento", current_timestamp())

)

nome_tabela_prata = "silver_transporte"
df_silver.write.format("delta").mode("overwrite").saveAsTable(nome_tabela_prata)


print("Camada Prata processada e salva com sucesso!!")
display(spark.sql(f"SELECT * FROM {nome_tabela_prata} LIMIT 10"))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7227616352490459>, line 19
      1 df_silver = (
      2     df_raw
      3     .withColumnRenamed("year", "ano")
   (...)
     15 
     16 )
     18 nome_tabela_prata = "silver_transporte"
---> 19 df_silver.write.format("delta").mode("overwrite").saveAsTable(nome_tabela_prata)
     22 print("Camada Prata processada e salva com sucesso!!")
     23 display(spark.sql(f"SELECT * FROM {nome_tabela_prata} LIMIT 10"))

File <command-7227616352490459>, line 2
      1 df_silver = (
----> 2     df_raw
      3     .withColumnRenamed("year", "ano")
      4     .withColumnRenamed('month', "mes")
      5     .withColumnRenamed("passagers", "qtd_passageiros")
      6     .dropna(subset=["ano", "mes", "qtd_passageiros"])
      7     .dropDuplicates()
      8     .withColumn(
      9         "categoria_demanda",
     10         when(col("

In [0]:
from pyspark.sql.functions import col, when, current_timestamp

# 1. Leitura direta da tabela Delta da Camada Bronze
df_raw = spark.read.table("bronze_transporte")

# 2. Etapa A: Renomeação e Limpeza inicial
df_cleaned = (
    df_raw
    .withColumnRenamed("year", "ano")
    .withColumnRenamed("month", "mes")
    .withColumnRenamed("passengers", "qtd_passageiros")
    .dropna()
    .dropDuplicates()
)

# 3. Etapa B: Regras de negócio e Metadados
df_silver = (
    df_cleaned
    .withColumn(
        "categoria_demanda",
        when(col("qtd_passageiros") >= 300, "Alta")
        .when(col("qtd_passageiros") >= 150, "Média")
        .otherwise("Baixa")
    )
    .withColumn("data_processamento", current_timestamp())
)

# 4. Gravando na Camada Prata como tabela Delta Lake
nome_tabela_prata = "silver_transporte"
df_silver.write.format("delta").mode("overwrite").saveAsTable(nome_tabela_prata)

# 5. Visualizando os dados tratados
print("Camada Prata criada com sucesso no Delta Lake!")
display(spark.sql(f"SELECT * FROM {nome_tabela_prata} LIMIT 10"))

Camada Prata criada com sucesso no Delta Lake!


ano,mes,qtd_passageiros,categoria_demanda,data_processamento
1949,September,136,Baixa,2026-09-18T14:56:30.736Z
1950,August,170,Média,2026-09-18T14:56:30.736Z
1950,September,158,Média,2026-09-18T14:56:30.736Z
1952,May,183,Média,2026-09-18T14:56:30.736Z
1953,August,272,Média,2026-09-18T14:56:30.736Z
1957,February,301,Alta,2026-09-18T14:56:30.736Z
1957,March,356,Alta,2026-09-18T14:56:30.736Z
1957,May,355,Alta,2026-09-18T14:56:30.736Z
1957,October,347,Alta,2026-09-18T14:56:30.736Z
1958,October,359,Alta,2026-09-18T14:56:30.736Z


In [0]:
from pyspark.sql.functions import sum, avg, round, count, desc

# 1. Leitura direta da tabela Delta tratada na Camada Prata
df_silver = spark.read.table("silver_transporte")

# 2. Agregação Executiva: Total e média de passageiros por ano
df_gold_anual = (
    df_silver
    .groupBy("ano")
    .agg(
        sum("qtd_passageiros").alias("total_passageiros"),
        round(avg("qtd_passageiros"), 2).alias("media_passageiros_mes"),
        count("mes").alias("total_meses_registrados")
    )
    .sort(desc("ano"))
)

# 3. Gravando na Camada Ouro como tabela Delta Lake de alta performance
nome_tabela_ouro = "gold_metricas_anuais"
df_gold_anual.write.format("delta").mode("overwrite").saveAsTable(nome_tabela_ouro)

# 4. Exibindo os dados prontos para consumo de BI
print("Camada Ouro construída e persistida com sucesso!")
display(spark.sql(f"SELECT * FROM {nome_tabela_ouro}"))

Camada Ouro construída e persistida com sucesso!


ano,total_passageiros,media_passageiros_mes,total_meses_registrados
1960,5714,476.17,12
1959,5140,428.33,12
1958,4572,381.0,12
1957,4421,368.42,12
1956,3939,328.25,12
1955,3408,284.0,12
1954,2867,238.92,12
1953,2700,225.0,12
1952,2364,197.0,12
1951,2042,170.17,12


In [0]:
# 1. Consultar o histórico de transações da tabela Ouro
df_historico = spark.sql("DESCRIBE HISTORY gold_metricas_anuais")

# 2. Exibir os detalhes de governança (versões, carimbos de data/hora e operações executadas)
display(df_historico.select("version", "timestamp", "operation", "operationParameters"))

version,timestamp,operation,operationParameters
0,2026-09-18T14:57:18.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)"
